# Simple RAG Agent with LangGraph

### Graph Architecture:
```
         __start__
             |
             v
          +-----+ 
          | llm |
          +-----+ 
         /       \ 
  True  /         \  False
       v           v
+-----------------+  __end__
| retriever_agent |
+-----------------+
       |
       +---> (loops back to llm)
```

### Workflow Explained:
1. **`__start__ -> llm`**: The user prompt enters the `llm` node with bound retrieval tools.
2. **Conditional Decision (`True` / `False`)**:
   - **`True`**: The LLM needs specific document context and generates a tool call $\rightarrow$ routes to **`retriever_agent`**.
   - **`False`**: The LLM has sufficient context to formulate the final answer or requires no retrieval $\rightarrow$ routes to **`__end__`**.
3. **`retriever_agent -> llm`**: The retriever tool executes similarity search across our knowledge base, returns relevant chunks as `ToolMessage`, and sends them back to `llm` to synthesize the answer.

## 0. Dependencies and Setup

In [ ]:
# Install dependencies if needed:
# !pip install langgraph langchain-core langchain-openai langchain-community python-dotenv

In [ ]:
import os
from typing import Annotated, Sequence, TypedDict, Literal
from dotenv import load_dotenv

# LangChain Core & OpenAI
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage,
    ToolMessage
)
from langchain_core.documents import Document
from langchain_core.tools import create_retriever_tool
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# LangGraph Components
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv()

## 1. Build Knowledge Base & Retriever Tool

We populate an in-memory vector store with company documents/knowledge about LangGraph architecture and policies, then convert it into a retriever tool.

In [ ]:
# Sample Knowledge Base Documents
knowledge_documents = [
    Document(
        page_content="LangGraph is a library designed for building stateful, multi-actor applications with LLMs. "
                     "It models agent workflows as graphs composed of Nodes (actions/computation) and Edges (transitions/conditions).",
        metadata={"topic": "langgraph_basics"}
    ),
    Document(
        page_content="In LangGraph, state persistence across turns is managed using Checkpointers such as MemorySaver or PostgresSaver. "
                     "Each execution thread is identified by a unique thread_id.",
        metadata={"topic": "persistence"}
    ),
    Document(
        page_content="The Acme Corp internal policy requires that all AI-generated drafts and code must pass automated unit testing "
                     "and receive explicit sign-off from a senior technical lead before deployment.",
        metadata={"topic": "company_policy"}
    ),
    Document(
        page_content="Acme Corp's primary customer support hotline is +1-800-555-0199 and the emergency escalation email is emergency@acmecorp.internal.",
        metadata={"topic": "support_contacts"}
    )
]

# Initialize Embeddings & Vector Store
# If OPENAI_API_KEY is not available in local test, we create a robust retriever
try:
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vector_store = InMemoryVectorStore.from_documents(knowledge_documents, embeddings)
    base_retriever = vector_store.as_retriever(search_kwargs={"k": 2})
    retriever_tool = create_retriever_tool(
        base_retriever,
        name="knowledge_base_retriever",
        description="Search and retrieve official Acme Corp documentation, policies, and LangGraph technical details."
    )
except Exception as e:
    # Fallback keyword-matching tool when embeddings cannot make live network calls
    from langchain_core.tools import tool
    @tool
    def knowledge_base_retriever(query: str) -> str:
        """Search and retrieve official Acme Corp documentation, policies, and LangGraph technical details."""
        query_lower = query.lower()
        matches = [doc.page_content for doc in knowledge_documents if any(word in doc.page_content.lower() for word in query_lower.split())]
        return "\n\n".join(matches) if matches else "No relevant documents found in knowledge base."
    retriever_tool = knowledge_base_retriever

tools = [retriever_tool]
print(f"Retriever Tool ready: '{retriever_tool.name}'")

## 2. Define State & Model Setup

State structure containing `messages` equipped with `add_messages` reducer to accumulate conversation and tool messages.

In [ ]:
class RAGState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# Initialize ChatOpenAI and bind retriever tool
llm = ChatOpenAI(model="gpt-4o", temperature=0.0)
llm_with_tools = llm.bind_tools(tools)

## 3. Node Definitions Matching Diagram (`llm` and `retriever_agent`)

- **`llm` node**: Evaluates current messages with knowledge base tool binding.
- **`retriever_agent` node**: Standard ToolNode executing the retriever tool.
- **Router**: Checks if `llm` generated `tool_calls` (`True` $\rightarrow$ `retriever_agent`, `False` $\rightarrow$ `__end__`).

In [ ]:
SYSTEM_PROMPT = SystemMessage(
    content="You are an expert technical assistant with access to a knowledge base retriever tool. "
            "Always use the knowledge_base_retriever tool to look up accurate company policies, technical details, "
            "and contact info before answering questions. Only answer once you have verified facts."
)

# Node 1: 'llm'
def llm_node(state: RAGState) -> dict:
    messages = list(state["messages"])
    if not messages or not isinstance(messages[0], SystemMessage):
        messages = [SYSTEM_PROMPT] + messages
        
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Node 2: 'retriever_agent' (Executes retriever tool)
retriever_agent_node = ToolNode(tools)

# Conditional Router matching image labels (True / False)
def should_retrieve(state: RAGState) -> Literal["True", "False"]:
    """
    Returns 'True' if tool_calls exist (needs retrieval),
    Returns 'False' if no tool_calls exist (ready to end).
    """
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "True"
    return "False"

## 4. Build the Graph Matching Diagram

Mapping the exact graph nodes & edges from the diagram:
- `START` (`__start__`) $\rightarrow$ `llm`
- `llm` --(`True`)--> `retriever_agent`
- `llm` --(`False`)--> `END` (`__end__`)
- `retriever_agent` $\rightarrow$ `llm` (loop back)

In [ ]:
builder = StateGraph(RAGState)

# 1. Add nodes with exact names from diagram
builder.add_node("llm", llm_node)
builder.add_node("retriever_agent", retriever_agent_node)

# 2. Connect START to llm
builder.add_edge(START, "llm")

# 3. Add conditional edges from llm (True -> retriever_agent, False -> END)
builder.add_conditional_edges(
    "llm",
    should_retrieve,
    {
        "True": "retriever_agent",
        "False": END
    }
)

# 4. Loop back from retriever_agent to llm
builder.add_edge("retriever_agent", "llm")

# Compile
rag_agent = builder.compile()
print("Simple RAG Agent compiled successfully!")

## 5. Visualizing the Graph Structure

Let's visualize the graph representation in ascii/mermaid.

In [ ]:
try:
    print(rag_agent.get_graph().draw_ascii())
except Exception as e:
    print("Graph structure:")
    print("START -> llm")
    print("llm -> [True: retriever_agent, False: END]")
    print("retriever_agent -> llm")

## 6. Query Execution & Verification

In [ ]:
def test_rag_query(user_query: str):
    print(f"\n{'='*75}")
    print(f"User Question: {user_query}")
    print(f"{'='*75}")
    
    try:
        result = rag_agent.invoke({"messages": [HumanMessage(content=user_query)]})
        print("\n--- Step-by-Step Trajectory ---")
        for msg in result["messages"]:
            if isinstance(msg, HumanMessage):
                print(f"[User]: {msg.content}")
            elif isinstance(msg, AIMessage):
                if getattr(msg, "tool_calls", None):
                    for tc in msg.tool_calls:
                        print(f"[LLM -> Tool Call]: {tc['name']}(query='{tc['args'].get('query', '')}')")
                else:
                    print(f"\n[Final Answer]:\n{msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f"[Retriever Tool Output]: {msg.content[:200]}...")
        print(f"{'='*75}\n")
    except Exception as e:
        print(f"Execution Note (OpenAI API key required for live queries): {e}")

In [ ]:
# Query requiring knowledge base retrieval
test_rag_query("What is Acme Corp's policy before deploying AI-generated code, and what is their emergency contact?")

In [ ]:
# Query requiring technical knowledge retrieval
test_rag_query("How does LangGraph manage state persistence across turns?")